In [1]:
import os
import cv2
import time
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

I0000 00:00:1779240476.218315   16672 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
os.environ["EGL_PLATFORM"] = "surfaceless" # Error Prevention untuk Linux


In [3]:
cap = cv2.VideoCapture(0)
if (cap.isOpened() == False):
    print("Error opening video stream or file")

In [4]:
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_rate = int(cap.get(cv2.CAP_PROP_FPS))

print("Frame Width: ", frame_width)
print("Frame Height: ", frame_height)
print("Frame Rate: ", frame_rate)

Frame Width:  640
Frame Height:  480
Frame Rate:  30


In [ ]:
# Video Codec for .MP4
codec = cv2.VideoWriter_fourcc(*'mp4v')

# Video File Name
filename = "dataset_video.mp4"

# Video Dimension
videodimension = (frame_width, frame_height)

In [6]:
out = cv2.VideoWriter(filename, 
                      codec, 
                      frame_rate, 
                      videodimension)

In [7]:
recording = False

while True:
    ret, frame = cap.read()

    if ret == False:
        print("Error retrieving frame")
        break

    if recording:
        cv2.putText(frame, "RECORDING [Tekan 'q' untuk stop]", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        # Jika status recording True, simpan frame ke dalam file mp4
        out.write(frame)
    else:
        cv2.putText(frame, "PREVIEW [Tekan 's' untuk record]", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
    cv2.imshow("Frame", frame)

    key = cv2.waitKey(1) & 0xFF
    
    if key == ord('s') and not recording:
        print("Recording Started")
        recording = True
        
    elif key == ord('q'):
        if recording:
            print("Recording Stopped")
            recording = False
        break


out.release()
cap.release()
cv2.destroyAllWindows()

QFontDatabase: Cannot find font directory /home/izzanns/GitHub_Repo/CNN_LSTM_SignLanguageModel/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/izzanns/GitHub_Repo/CNN_LSTM_SignLanguageModel/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/izzanns/GitHub_Repo/CNN_LSTM_SignLanguageModel/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/izzanns/GitHub_Repo/CNN_LSTM_SignLanguageModel/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (fro

Recording Started


[ WARN:0@40.233] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.301] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.369] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.431] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.499] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.565] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.633] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.701] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.764] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.832] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.897] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@40.963] global cap_ffmpeg.cpp:218 write FFmpeg: Failed to write frame
[ WARN:0@41.033] global cap_ffmpeg.cpp:218 write FFm

Recording Stopped


In [10]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Inisialisasi utilities untuk menggambar (drawing utils)
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Konfigurasi Hand Landmarker dari Tasks API
base_options = python.BaseOptions(model_asset_path='handlandmarker/hand_landmarker.task')

# Mengatur RunningMode ke VIDEO karena kita membaca dari file video
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2)

# Buka file dataset video yang kita buat sebelumnya
video_path = 'dataset_video.mp4'
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

if not cap.isOpened():
    print(f"Error: Tidak dapat membuka video {video_path}")
else:
    # Buat instance detector untuk hand landmarking
    with vision.HandLandmarker.create_from_options(options) as detector:
        frame_index = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                print("Selesai memutar video dataset.")
                break
                
            # OpenCV membaca format BGR, konversi ke RGB untuk MediaPipe
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Format frame ke dalam MediaPipe Image
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            
            # Hitung timestamp dalam satuan ms
            timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))
            if timestamp_ms == 0 and frame_index > 0:
                timestamp_ms = int(frame_index * 1000 / fps)
                
            frame_index += 1
            
            # Lakukan deteksi / hand landmarking
            result = detector.detect_for_video(mp_image, timestamp_ms)
            
            # Jika ditemukan tangan, gambar hasil landmarknya
            if result.hand_landmarks:
                for hand_landmarks in result.hand_landmarks:
                    # Konversi hasil format Tugas API ke format Proto untuk drawing util
                    hand_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
                    hand_landmarks_proto.landmark.extend([
                        landmark_pb2.NormalizedLandmark(x=landmark.x, y=landmark.y, z=landmark.z) 
                        for landmark in hand_landmarks
                    ])
                    
                    # Gambarkan titik landmark dan koneksi jari di frame asli (BGR)
                    mp_drawing.draw_landmarks(
                        frame,
                        hand_landmarks_proto,
                        mp_hands.HAND_CONNECTIONS,
                        mp_drawing_styles.get_default_hand_landmarks_style(),
                        mp_drawing_styles.get_default_hand_connections_style())
            
            # Tampilkan frame yang sudah diberi anotasi dengan OpenCV
            cv2.imshow('Hand Landmarking pada Dataset', frame)
            
            # Tekan tombol 'q' kapan saja untuk keluar dari preview
            # Pengaturan delay ini juga disesuaikan dengan FPS video asli
            delay = int(1000/fps) if fps > 0 else 30
            if cv2.waitKey(delay) & 0xFF == ord('q'):
                break
                
    cap.release()
    cv2.destroyAllWindows()

Error: Tidak dapat membuka video dataset_video.mp4
